# Study 2 — fine-tuning T5 for log summarization, then serving it

This study takes a model that already exists — `google-t5/t5-small`, 60M parameters —
and teaches it a job: turn a raw infrastructure log line into a one-sentence summary.

**Where this comes from.** The production ancestor lived in an alerting pipeline
drowning in near-duplicate alerts: the same failure arrives as a thousand
slightly-different log lines (different timestamps, hosts, request ids), so naive
deduplication misses almost everything. Summarization normalizes the lines to their
*meaning*; content-hashing the summaries then collapses the storm — most of the alert
redundancy in that system. A small fine-tuned model mattered: per-alert inference must
be cheap, self-hosted, and fast, which rules out calling a large general model per line.

The study has three parts, and all three are the point:

1. **The recipe** — small, boring, correct. Most fine-tuning failures are recipe bugs,
   not model choices.
2. **The evaluation** — base vs fine-tuned on a held-out split, so the delta is the
   fine-tune's contribution, measured, not asserted.
3. **The serving** — a fine-tuned model is useless until something answers HTTP. A
   semaphore-bounded replica pool serves it without letting one slow generate() call
   melt the event loop.

In [1]:
import pandas as pd

corpus = pd.read_csv("data/log_summary_pairs.csv")
print(f"{len(corpus)} pairs")
print(f"log length:     median {corpus['log'].str.split().str.len().median():.0f} words, "
      f"p95 {corpus['log'].str.split().str.len().quantile(0.95):.0f}")
print(f"summary length: median {corpus['summary'].str.split().str.len().median():.0f} words, "
      f"p95 {corpus['summary'].str.split().str.len().quantile(0.95):.0f}")
corpus.sample(3, random_state=7)

4000 pairs
log length:     median 13 words, p95 18
summary length: median 15 words, p95 20


,log,summary
2060,Message on payments.retry was redelivered 25 t...,A message on payments.retry exhausted 25 redel...
933,WARN service=feed-aggregator endpoint=/api/v1/...,Rate limiting kicked in on /api/v1/search — ov...
378,Request to /api/v1/uploads rejected with 401: ...,An expired token (463 minutes past validity) c...


The corpus is **generated, not collected**: 4,000 unique pairs from the seeded template
grammar in `src/model_forge/summarizer/corpus.py`, spanning 14 infrastructure failure
domains (container OOMs, database errors, TLS expiry, queue dead-letters, ...). Each
domain renders through several log shapes — terse key-value lines, bracketed logger
output, prose — while the paired summary always restates component, cause, and code in
fluent prose. Many surface forms mapping to one normalized statement is exactly the
normalization job the production ancestor needed. And because every byte comes from the
generator, provenance is trivial: no source dataset, nothing to leak, byte-for-byte
reproducible via `python -m model_forge.summarizer.corpus`.

## The recipe

Everything below is in `src/model_forge/summarizer/train.py`; the notebook just explains
the choices.

| Choice | Value | Why |
|---|---|---|
| Task prefix | `summarize: ` | T5 is text-to-text: the prefix **is** the task. It must match at train and inference — `model.py` owns it so it can't diverge. |
| Truncation | 256 in / 128 out | p95 log is far shorter; longer budgets just pad. |
| Label masking | pads → −100 | Otherwise the loss rewards predicting padding. |
| LR | 5e-5, linear decay, 500 warmup | The T5 fine-tuning workhorse setting. |
| Epochs | 5, early stop patience 3 | Small model + small corpus overfits fast; eval-loss checkpointing keeps the best epoch, not the last. |
| Seed | 1337 | The 90/10 split is deterministic — evaluation below sees only never-trained pairs. |

In [2]:
RUN_TRAINING = False  # flip to True to fine-tune here (~15 min on Apple MPS, ~40 min CPU)

if RUN_TRAINING:
    from model_forge.summarizer.train import train
    train(epochs=5, batch_size=16)
else:
    print("Using the canonical run: python -m model_forge.summarizer.train --epochs 5")
    print("Model saved to models/log-summarizer-t5/")

Using the canonical run: python -m model_forge.summarizer.train --epochs 5
Model saved to models/log-summarizer-t5/


## Did the fine-tune actually help?

`evaluate.py` generates summaries for all 400 held-out pairs twice — once with stock
`t5-small`, once with the fine-tune — and scores both against the references with ROUGE.
The base model's zero-shot behavior on the `summarize:` prefix is the honest baseline:
that's exactly what you'd deploy if you skipped fine-tuning.

In [3]:
import json

rouge = json.load(open("docs/summarizer/rouge.json"))
table = pd.DataFrame([
    {"model": "t5-small (zero-shot)", **rouge["base"]},
    {"model": "fine-tuned", **rouge["fine_tuned"]},
]).set_index("model")
print(f"held-out pairs: {rouge['eval_pairs']}")
table

held-out pairs: 400


,rouge1,rouge2,rougeLsum
model,,,
t5-small (zero-shot),0.4591,0.1834,0.3265
fine-tuned,0.6022,0.3384,0.4505


In [4]:
from model_forge.summarizer.model import SummarizationModel
from model_forge.summarizer.train import load_pairs

_, eval_df = load_pairs("data/log_summary_pairs.csv")
samples = eval_df.head(3)

base = SummarizationModel("google-t5/t5-small")
tuned = SummarizationModel("models/log-summarizer-t5")

for _, row in samples.iterrows():
    print("LOG:      ", row["log"][:160])
    print("reference:", row["summary"][:160])
    print("base:     ", base.summarize(row["log"])[:160])
    print("tuned:    ", tuned.summarize(row["log"])[:160])
    print("-" * 80)

/private/tmp/claude-501/-Users-hassansaleem-Desktop-github/c69f9c47-eb61-4b99-a72f-bc8deadb54ae/scratchpad/modelforge/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 13778.02it/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 19180.15it/s]

LOG:       [CRIT] node-08: volume /var/log at 97%; writes from auth-gateway are failing in connectDb
reference: Writes from auth-gateway on node-08 are failing because volume /var/log is 97% full — connectDb hit 'no space left on device'.
base:      connectDb is failing to connectDb.
tuned:     volume /var/log on node-08 is 97% higher than auth-gateway's connectDb.
--------------------------------------------------------------------------------
LOG:       [CRIT] payment-worker: 401 on /api/v1/orders — expired credentials (157m past validity)
reference: An expired token (157 minutes past validity) caused payment-worker to reject /api/v1/orders with 401.


base:      expired credentials (157m past validity) expired credentials.
tuned:     Payment-worker failed to process /api/v1/orders with 401 (exit 157 minutes past validity).
--------------------------------------------------------------------------------
LOG:       FATAL service=media-encoder fn=acquireLock missing_env=MAX_WORKERS exit=1
reference: A missing MAX_WORKERS value stopped media-encoder from booting (acquireLock exited 1).
base:      fn=acquireLock missing_env=MAX_WORKERS exit=1.


tuned:     Media-encoder failed to start because the MAX_WORKERS database address was missing; the acquireLock call failed with exit code 1.
--------------------------------------------------------------------------------


The base model mostly extracts a prefix of the input — zero-shot `t5-small` has no idea
what matters in a log line. The fine-tune writes an actual summary: error code, failing
function, cause. More side-by-side pairs live in
[docs/summarizer/examples.md](../docs/summarizer/examples.md).

## Serving it

A transformer's `generate()` is blocking, GIL-bound-ish, and not re-entrant — one model
instance behind an async endpoint serializes every request, and an unbounded executor
piles requests onto a busy model. The serving layer
(`src/model_forge/summarizer/pool.py` + `api.py`) does three specific things:

1. **N replicas behind a semaphore** — at most N requests run inference at once;
   round-robin hands each admitted request a different replica; the rest queue.
2. **The event loop never runs a forward pass** — generation happens in a thread pool
   via `run_in_executor`.
3. **Health checks probe, they don't queue** — `/health_check` uses `try_acquire()`
   (probe capacity, don't hold it): a saturated pool answers **503** so an orchestrator
   sheds load instead of queueing its liveness probe behind a slow generate.

```bash
MODEL_DIR=models/log-summarizer-t5 POOL_SIZE=2 python bin/serve.py
curl -s localhost:8000/summarize -H 'content-type: application/json' \
  -d '{"text": "ERROR Failed to connect to database. Function connectDB. Access denied."}'
```

The same pattern shipped the production model: the pool bound is what let one modest
box absorb alert storms with predictable latency instead of collapsing.

## What transfers

The recipe, the base-vs-tuned evaluation discipline, and the pooled serving design are
dataset-agnostic. Swap the corpus and the same three files fine-tune and serve any
seq2seq task — the log domain here is just the one where I first needed it.